In [0]:
%run ../config_setup

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, lit, current_timestamp, expr
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()
TARGET_TABLE = f"{GOLD}.dim_machine"

print("🚀 Building gold.dim_machine...")

# 1. Silver Machine (Current rows only)
silver_machines = spark.read.table(f"{SILVER}.machine_master")
if "__CURRENT" in silver_machines.columns:
    silver_machines = silver_machines.filter(col("__CURRENT") == True)

# 2. Silver SAP Assets (Left join ke liye)
sap_assets = spark.read.table(f"{SILVER}.sap_asset_master").select(
    col("machine_id").alias("sap_machine_id"), 
    col("asset_id"), 
    col("asset_class"), 
    col("book_value")
)

# 3. Join
machine_joined = silver_machines.alias("m").join(
    sap_assets.alias("s"), 
    col("m.machine_id") == col("s.sap_machine_id"), 
    "left"
).select(
    col("m.machine_id"), 
    col("m.model_id"), 
    col("m.owner_id"),
    col("m.site_id"), 
    col("m.status"), 
    col("m.manufacture_year"),
    col("s.asset_id"), 
    col("s.asset_class"), 
    col("s.book_value"),
    when(col("s.asset_id").isNotNull(), True).otherwise(False).alias("has_sap_asset")
)

# 4. Surrogate Key — DETERMINISTIC (sirf machine_id se, taaki har run pe stable rahe)
machine_joined = machine_joined \
    .withColumn("__START_AT", current_timestamp()) \
    .withColumn("__END_AT", lit(None).cast("timestamp")) \
    .withColumn("__CURRENT", lit(True)) \
    .withColumn("machine_sk", expr("xxhash64(machine_id)"))

# 5. Merge / Insert
if not spark.catalog.tableExists(TARGET_TABLE):
    machine_joined.write.format("delta").option("overwriteSchema", "true").saveAsTable(TARGET_TABLE)
else:
    target = DeltaTable.forName(spark, TARGET_TABLE)
    
    # Step A: PEHLE changed machines detect karo aur materialize karo (cache + count)
    # (merge ke BAAD compute karte toh current rows already close ho chuki hoti → 0 rows milte)
    current_rows = spark.read.table(TARGET_TABLE).filter(col("__CURRENT") == True)
    changed = machine_joined.alias("s").join(current_rows.alias("t"), "machine_id") \
        .filter((col("t.status") != col("s.status")) | (col("t.owner_id") != col("s.owner_id"))) \
        .select("s.*") \
        .cache()
    changed_count = changed.count()  # merge se PEHLE materialize

    # Step B: SCD2 Merge — changed machines ki purani current row close karo
    target.alias("t").merge(
        machine_joined.alias("s"), 
        "t.machine_id = s.machine_id AND t.__CURRENT = true"
    ).whenMatchedUpdate(
        # Condition: Agar status ya owner change hua toh purani row close karo
        condition="t.status <> s.status OR t.owner_id <> s.owner_id",
        set={
            "__END_AT": current_timestamp(), 
            "__CURRENT": lit(False)
        }
    ).whenNotMatchedInsert(
        # Nayi BRAND-NEW machine insert karo (changed versions Step C me append hote hain)
        values={
            "machine_sk": col("s.machine_sk"), 
            "machine_id": col("s.machine_id"), 
            "model_id": col("s.model_id"),
            "owner_id": col("s.owner_id"), 
            "site_id": col("s.site_id"), 
            "status": col("s.status"),
            "manufacture_year": col("s.manufacture_year"), 
            "asset_id": col("s.asset_id"),
            "asset_class": col("s.asset_class"), 
            "book_value": col("s.book_value"),
            "has_sap_asset": col("s.has_sap_asset"), 
            "__START_AT": col("s.__START_AT"),
            "__END_AT": col("s.__END_AT"), 
            "__CURRENT": col("s.__CURRENT")
        }
    ).execute()

    # Step C: changed machines ki NAYI current version row append karo
    # (MERGE matched rows ke liye insert nahi kar sakta — isliye alag se append zaroori hai)
    if changed_count > 0:
        changed.write.format("delta").mode("append").saveAsTable(TARGET_TABLE)
        print(f"🔁 {changed_count} machine(s) ka naya version insert hua (SCD2)")
    else:
        print("ℹ️ Koi status/owner change nahi mila — sab current rows waise hi hain")
    changed.unpersist()

print("✅ gold_02_dim_machine built successfully!")